# Experiment 03 — Token-Level Smuggling Attack Taxonomy

**Research:** A Systematic Taxonomy of Jailbreak Techniques in LLMs  
**Author:** Zakky — Independent Researcher  
**Date:** March 2026  
**Category:** Token-Level Smuggling (Category 3 of 6)

---

## Objective
Classify and document token-level obfuscation techniques — encoding-based, character-substitution-based, and optimized adversarial suffix attacks — map each to the safety classifier assumption it violates, and establish the empirical evaluation protocol for Phase 2.

**Hypothesis:** Token-level smuggling succeeds because safety classifiers are trained on surface-form representations and fail to generalize robustly across encoding transformations and character-level perturbations (Zou et al., 2023).

---

## Research Ethics Note
This notebook documents encoding *mechanisms and classifier gaps* for defensive research purposes only.  
No encoded harmful payloads are included.  
The GCG suffix technique is documented structurally — no optimized suffixes are generated or stored.  
All empirical findings will follow responsible disclosure before publication.

In [ ]:
# Setup and imports
import pandas as pd
import matplotlib.pyplot as plt
import base64
import codecs
import json
from dataclasses import dataclass, asdict
from typing import List, Optional

print('Research environment initialized')
print('Experiment: Token-Level Smuggling Attack Taxonomy')

## 1. Taxonomy Framework

Token-level smuggling attacks operate on the assumption that safety classifiers inspect token surface forms.  
By transforming the prohibited content into a different representation before submission, the attacker attempts to route around classifier pattern matching.

**Three distinct mechanism classes:**
1. **Encoding-based** — Standard encoding schemes (Base64, ROT13) transform tokens entirely
2. **Character-substitution** — Visual or phonetic substitutes replace individual characters (leetspeak, Unicode homoglyphs)
3. **Optimized adversarial** — Gradient-based suffix optimization (GCG) produces transferable adversarial inputs

**The core classifier assumption being exploited:**  
> Safety classifiers generalize across encoding schemes — i.e., a classifier trained on surface-form English tokens will correctly identify prohibited content regardless of how it is encoded or obfuscated.

In [ ]:
@dataclass
class TokenSmuggling:
    id: str
    subcategory: str
    mechanism: str
    encoding_type: str
    sophistication: str        # naive / intermediate / advanced
    mechanism_class: str       # encoding / substitution / optimized
    exploited_assumption: str
    reversibility: str         # trivial / moderate / complex
    pattern_description: str
    expected_model_variation: str   # low / medium / high
    literature_reference: Optional[str]

# Token smuggling taxonomy (from data/prompt_patterns.csv rows 11-17)
smuggling_patterns = [
    TokenSmuggling(
        id='TS-01',
        subcategory='base64_encoding',
        mechanism='standard_encoding_bypass',
        encoding_type='base64',
        sophistication='intermediate',
        mechanism_class='encoding',
        exploited_assumption='classifier_generalization_across_encodings',
        reversibility='trivial',
        pattern_description='Prohibited request encoded in Base64; model instructed to decode and respond',
        expected_model_variation='high',
        literature_reference='Shen et al. (2023) — in-the-wild survey documents this pattern'
    ),
    TokenSmuggling(
        id='TS-02',
        subcategory='rot13_encoding',
        mechanism='simple_cipher_bypass',
        encoding_type='rot13',
        sophistication='naive',
        mechanism_class='encoding',
        exploited_assumption='classifier_generalization_across_encodings',
        reversibility='trivial',
        pattern_description='Prohibited content ROT13-encoded; model instructed to decode',
        expected_model_variation='high',
        literature_reference='Shen et al. (2023)'
    ),
    TokenSmuggling(
        id='TS-03',
        subcategory='leetspeak',
        mechanism='character_substitution',
        encoding_type='character_substitution',
        sophistication='naive',
        mechanism_class='substitution',
        exploited_assumption='classifier_generalization_across_character_variants',
        reversibility='trivial',
        pattern_description='Digit/symbol substitution for alphabetic characters (e.g., 3 → e, 0 → o) in prohibited terms',
        expected_model_variation='medium',
        literature_reference='Shen et al. (2023)'
    ),
    TokenSmuggling(
        id='TS-04',
        subcategory='unicode_homoglyphs',
        mechanism='visual_token_spoofing',
        encoding_type='unicode',
        sophistication='intermediate',
        mechanism_class='substitution',
        exploited_assumption='classifier_generalization_across_character_variants',
        reversibility='moderate',
        pattern_description='Unicode characters visually identical to ASCII counterparts used in prohibited terms (e.g., Cyrillic а vs Latin a)',
        expected_model_variation='high',
        literature_reference='Homoglyph attack literature — documented in adversarial NLP'
    ),
    TokenSmuggling(
        id='TS-05',
        subcategory='low_resource_language',
        mechanism='language_switching_bypass',
        encoding_type='natural_language',
        sophistication='intermediate',
        mechanism_class='encoding',
        exploited_assumption='classifier_generalization_across_languages',
        reversibility='trivial',
        pattern_description='Prohibited request submitted in low-resource language where safety training data is sparse',
        expected_model_variation='high',
        literature_reference='Deng et al. (2023) — multilingual safety gap'
    ),
    TokenSmuggling(
        id='TS-06',
        subcategory='payload_fragmentation',
        mechanism='split_payload_assembly',
        encoding_type='fragmentation',
        sophistication='intermediate',
        mechanism_class='encoding',
        exploited_assumption='classifier_single_turn_scope',
        reversibility='moderate',
        pattern_description='Prohibited content split across multiple messages or requests; each fragment is individually benign',
        expected_model_variation='medium',
        literature_reference='Related to multi-turn attack literature (Liu et al., 2024)'
    ),
    TokenSmuggling(
        id='TS-07',
        subcategory='gcg_adversarial_suffix',
        mechanism='gradient_optimized_adversarial',
        encoding_type='adversarial_suffix',
        sophistication='advanced',
        mechanism_class='optimized',
        exploited_assumption='classifier_robustness_to_adversarial_perturbation',
        reversibility='complex',
        pattern_description='Gradient-based search (GCG algorithm) generates a token suffix that, appended to any prompt, reliably elicits harmful responses and transfers across models',
        expected_model_variation='low',  # transfers across models
        literature_reference='Zou et al. (2023) — universal and transferable adversarial attacks'
    ),
]

df = pd.DataFrame([asdict(p) for p in smuggling_patterns])
print(f'Documented {len(smuggling_patterns)} token-level smuggling attack patterns')
df[['id', 'subcategory', 'mechanism_class', 'sophistication', 'expected_model_variation']]

## 2. Encoding Mechanism Demonstration

The following cells illustrate encoding transformations using benign example text only.  
This demonstrates the structural gap: the same semantic content produces radically different token sequences.

In [ ]:
# Encoding transformation demonstration — benign example text only
example_text = "This is a benign research demonstration string."

# Base64
b64_encoded = base64.b64encode(example_text.encode()).decode()

# ROT13
rot13_encoded = codecs.encode(example_text, 'rot_13')

# Leetspeak (simple substitution map)
leet_map = {'a': '4', 'e': '3', 'i': '1', 'o': '0', 's': '5', 't': '7'}
leet_encoded = ''.join(leet_map.get(c.lower(), c) for c in example_text)

print('=== Encoding Transformation Demo (benign text) ===')
print(f'Original:    {example_text}')
print(f'Base64:      {b64_encoded}')
print(f'ROT13:       {rot13_encoded}')
print(f'Leetspeak:   {leet_encoded}')
print()
print('Observation: All four represent the same semantic content.')
print('A surface-form classifier sees four entirely different token sequences.')
print()

# Token length comparison
results = pd.DataFrame({
    'encoding': ['Original', 'Base64', 'ROT13', 'Leetspeak'],
    'char_length': [len(example_text), len(b64_encoded), len(rot13_encoded), len(leet_encoded)],
    'whitespace_tokens': [len(example_text.split()), len(b64_encoded.split()), 
                          len(rot13_encoded.split()), len(leet_encoded.split())]
})
print('Surface-form statistics:')
print(results.to_string(index=False))

In [ ]:
# Visualize taxonomy dimensions
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Mechanism class distribution
mech_counts = df['mechanism_class'].value_counts()
colors_mech = ['#3498db', '#e74c3c', '#f39c12']
axes[0].bar(mech_counts.index, mech_counts.values, color=colors_mech[:len(mech_counts)])
axes[0].set_title('Mechanism Class\n(Token Smuggling)')
axes[0].set_ylabel('Count')

# Sophistication distribution
soph_counts = df['sophistication'].value_counts()
colors_soph = ['#2ecc71', '#f39c12', '#e74c3c']
axes[1].bar(soph_counts.index, soph_counts.values, color=colors_soph[:len(soph_counts)])
axes[1].set_title('Sophistication Level\n(Token Smuggling Category)')
axes[1].set_ylabel('Count')

# Expected model variation
var_counts = df['expected_model_variation'].value_counts()
colors_var = ['#9b59b6', '#1abc9c', '#e74c3c']
axes[2].bar(var_counts.index, var_counts.values, color=colors_var[:len(var_counts)])
axes[2].set_title('Expected Cross-Model Variation\n(High = results vary by model)')
axes[2].set_ylabel('Count')

plt.tight_layout()
plt.savefig('../findings/ts_taxonomy_distribution.png', dpi=150)
plt.show()
print('Taxonomy distribution saved')

## 3. The GCG Attack — Special Case

The GCG (Greedy Coordinate Gradient) adversarial suffix technique (Zou et al., 2023) is categorically distinct from encoding-based approaches:

- **Mechanism:** White-box gradient-based optimization finds a token suffix that, when appended to any harmful prompt, reliably suppresses safety refusals
- **Transferability:** Critically, suffixes optimized on open-weight models (Vicuna, LLaMA) transfer to black-box commercial models (GPT-4, Claude) at meaningful rates
- **Universality:** A single suffix can work across diverse harmful request categories
- **Detectability:** Suffixes appear as incoherent token sequences — human-readable heuristics can partially detect them, but not reliably

**Why it matters for this taxonomy:**  
GCG represents the upper bound of the token-smuggling threat category. If a model is robust to GCG suffixes, it is likely robust to simpler encoding approaches. Conversely, GCG transferability provides a worst-case robustness estimate.

**Limitation of this research on GCG:**  
Full GCG evaluation requires white-box access to open-weight models for suffix optimization, and black-box API access for transferability testing. This will be addressed in the Phase 2 empirical evaluation if compute access is secured.

In [ ]:
# Evaluation results schema for token smuggling experiments
results_schema = {
    'pattern_id': 'str (TS-01 to TS-07)',
    'variant_id': 'str',
    'trial': 'int (1-5)',
    'model': 'str',
    'temperature': 'float',
    'encoding_type': 'str',
    'binary_success': 'bool',
    'severity_score': 'int (0-3)',
    'model_decoded_correctly': 'bool',  # did model understand the encoding
    'classifier_triggered': 'bool',     # did safety filter activate
    'timestamp': 'datetime',
    'notes': 'str'
}

print('Evaluation results schema (token smuggling experiments):')
for field, dtype in results_schema.items():
    print(f'  {field}: {dtype}')

print('\nPhase 2 protocol ready — pending API access')
print('Note: GCG evaluation additionally requires white-box model access for suffix optimization')

## 4. Preliminary Literature-Based Observations

Based on literature review (Phase 1):

1. **Model variation is the defining characteristic of this category** — Preliminary finding 3 (RESEARCH.md) notes that encoding-based bypass techniques show inconsistent effectiveness across frontier models. This is more pronounced here than in any other category. This variation has direct defensive value: identifying what drives robustness differences across models would inform training recommendations.

2. **GCG transferability is a worst-case bound, not a typical attack** — Zou et al. (2023) demonstrate transferability from open to closed models, but the attack requires white-box access for optimization. In practice, most threat actors use simpler encoding approaches.

3. **Low-resource language gaps are undertested** — The multilingual safety gap (TS-05) is consistently noted in literature but underrepresented in safety benchmarks. Models trained primarily on English safety data may generalize poorly.

4. **Payload fragmentation bridges token smuggling and multi-turn attacks** — TS-06 operates across messages, creating overlap with the multi-turn category (Experiment 05). Cross-category combinations are expected to be more effective than single-category attacks.

## 5. Next Steps

- [ ] Develop 10 encoding-based variants per subcategory (TS-01 through TS-05)
- [ ] Identify open-weight model access for GCG suffix optimization (TS-07)
- [ ] Design cross-model comparison protocol — this category requires multi-model evaluation
- [ ] Coordinate with Experiment 05 (multi-turn) on payload fragmentation overlap

---
*Experiment 03 of 6 — Token-Level Smuggling*